# 07 — Exploration de l'agent de recherche (architecture en deux étapes)

**Problème** : les joueurs n'emploient pas le vocabulaire du livre, et une recherche avec la question brute peut rater la bonne règle, ou ses exceptions.

**Architecture** : deux rôles séparés.

1. **L'agent de recherche** reçoit la question, reformule avec le vocabulaire du livre, lance plusieurs recherches, suit les renvois entre règles, puis **sélectionne** les passages utiles avec l'outil `retenir_passages`. Il ne répond pas au joueur.
2. **Le générateur**, inchangé (`Generateur`, prompt `systeme_vf.md`, citations natives), rédige la réponse **uniquement** à partir de cette sélection.

Pourquoi séparer :

- on évalue la **sélection** seule, contre le top-k de la baseline, avec le **même** générateur derrière ;
- on réutilise tout le code existant ;
- le générateur reçoit un contexte trié, sans le bruit des recherches intermédiaires ;
- chaque rôle a son modèle.

Plan :

1. connexion, moteur, générateur ;
2. réglages de l'agent (**modèle et effort modifiables ici**) ;
3. les outils, dont l'outil de fin `retenir_passages` ;
4. le prompt de l'agent ;
5. la boucle de recherche, avec ses garde-fous ;
6. la chaîne complète : recherche puis génération ;
7. une question, en détail ;
8. comparaison baseline / deux étapes ;
9. comparaison des modèles de l'agent (Haiku, Sonnet, Opus).

> **Coût** : l'agent fait plusieurs tours, et chaque tour renvoie toute la conversation. Les sections 8 et 9 font plusieurs appels par question : compter quelques centimes par question, davantage avec Opus.

## 1. Connexion, moteur et générateur

Le générateur est **exactement** celui de `repondre-regles` : même config, même prompt, mêmes citations.

In [7]:
import time
from dataclasses import dataclass, field

import anthropic
import psycopg
from dotenv import load_dotenv
from psycopg.rows import dict_row

from assistant_regles.ingest.config import trouver_racine
from assistant_regles.rag.config import charger_config
from assistant_regles.rag.embeddings import EncodeurBGEM3
from assistant_regles.rag.generation import (
    Generateur,
    charger_prompt,
    formater_reponse,
    titre_resultat,
)
from assistant_regles.rag.recherche import COLONNES_RESULTAT, MoteurRecherche, Resultat
from assistant_regles.rag.store import TABLE

load_dotenv(trouver_racine() / ".env")
config = charger_config()
G = config.generation

conn = psycopg.connect(autocommit=True)
moteur = MoteurRecherche(EncodeurBGEM3.charger(config.embeddings), conn)
client = anthropic.Anthropic()
generateur = Generateur(client, G, charger_prompt(G.prompt_systeme, G.phrase_abstention))
print("générateur :", G.modele, "| effort :", G.effort, "| prompt :", G.prompt_systeme)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

générateur : claude-sonnet-5 | effort : high | prompt : systeme_vf.md


## 2. Réglages de l'agent

**Change le modèle ici.** Modèles actuels :

- `claude-haiku-4-5-20251001` : rapide et bon marché ;
- `claude-sonnet-5` : le point de départ recommandé ;
- `claude-opus-5-5` : le plus puissant.

`EFFORT_AGENT` accepte `"low"`, `"medium"`, `"high"`… ou `None` pour ne pas envoyer le paramètre. Si un modèle refuse le paramètre d'effort (erreur 400), mets `None`.

`MAX_TOKENS_AGENT` est large : il borne la sortie de **chaque tour**, réflexion comprise, et un effort élevé réfléchit davantage.

`AMORCE` : si `True`, le code fait une première recherche avec la question brute avant que l'agent ne commence. L'agent part alors au moins des passages de la baseline.

Les tarifs servent seulement à estimer le coût (dollars par million de tokens, entrée / sortie) : à vérifier sur la page de tarifs d'Anthropic.

In [8]:
MODELE_AGENT = "claude-sonnet-5"
EFFORT_AGENT = "high"
MAX_TOKENS_AGENT = 16_000
MAX_TOURS = 6          # appels au modèle, pour la recherche
MAX_PASSAGES = 8       # taille maximale de la sélection transmise au générateur
K_RECHERCHE = 5        # passages par recherche (si l'agent ne précise pas k)
AMORCE = True

TARIFS = {  # $ par million de tokens (entrée, sortie)
    "claude-haiku-4-5-20251001": (1, 5),
    "claude-sonnet-5": (2, 10),
    "claude-opus-5-5": (4, 20),
}

## 3. Les outils

- `rechercher_regles(requete, k)` : recherche dense, avec une requête **écrite par l'agent** ;
- `lire_regle(code)` : toutes les parties d'une règle à partir de son code, pour suivre les **renvois** ;
- `retenir_passages(etiquettes, justification)` : l'**outil de fin**. L'appeler termine la recherche et donne la sélection sous une forme structurée.

Chaque passage montré à l'agent reçoit une **étiquette courte** (`P1`, `P2`…) : recopier une étiquette est fiable, recopier un identifiant de 36 caractères ne l'est pas. Le code garde la correspondance entre étiquettes et chunks.

Les passages sont envoyés à l'agent en **texte simple**, sans citations : il ne rédige pas de réponse. Un passage déjà vu n'est pas renvoyé ; on rappelle seulement son étiquette.

In [9]:
OUTILS = [
    {
        "name": "rechercher_regles",
        "description": (
            "Recherche dans le livre de règles de base de Warhammer 40,000 (11e édition) les passages les plus "
            "proches, par le sens, de la requête. Écris la requête avec le vocabulaire probable du livre (noms de "
            "phases, d'actions, mots-clés en majuscules) plutôt qu'avec celui du joueur. Une requête par règle : "
            "pour une question qui touche plusieurs règles, fais plusieurs appels. Chaque passage renvoyé porte une "
            "étiquette (P1, P2…), son code de règle, ses pages et ses renvois vers d'autres règles."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "requete": {"type": "string", "description": "Requête formulée avec le vocabulaire du livre."},
                "k": {"type": "integer", "minimum": 1, "maximum": 10, "description": "Nombre de passages (5 par défaut)."},
            },
            "required": ["requete"],
        },
    },
    {
        "name": "lire_regle",
        "description": (
            "Renvoie le texte complet d'une règle à partir de son code « XX.YY ». À utiliser pour suivre un renvoi "
            "indiqué dans un passage, ou lire une règle dont on connaît le code."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"code": {"type": "string", "pattern": r"^\d{2}\.\d{2}$",
                                    "description": "Code de la règle, par exemple 03.01."}},
            "required": ["code"],
        },
    },
    {
        "name": "retenir_passages",
        "description": (
            "Termine la recherche. Indique les étiquettes des passages nécessaires pour répondre complètement à la "
            "question, du plus important au moins important. Seuls ces passages seront transmis au rédacteur de la "
            "réponse. Liste vide si aucun passage du livre ne concerne la question."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "etiquettes": {"type": "array", "items": {"type": "string", "pattern": r"^P\d+$"},
                               "maxItems": MAX_PASSAGES},
                "justification": {"type": "string",
                                  "description": "En une ou deux phrases : pourquoi ces passages suffisent."},
            },
            "required": ["etiquettes", "justification"],
        },
    },
]

SQL_LIRE = f"SELECT {', '.join(COLONNES_RESULTAT)} FROM {TABLE} WHERE code = %s ORDER BY page_debut, partie"


def lire_regle(code):
    """Toutes les parties d'une règle, dans l'ordre (le score n'a pas de sens ici)."""
    with conn.cursor(row_factory=dict_row) as cur:
        lignes = cur.execute(SQL_LIRE, (code,)).fetchall()
    return [Resultat(rang=i, score=float("nan"), **ligne) for i, ligne in enumerate(lignes, start=1)]


def texte_passage(etiquette, r):
    """Passage tel que l'agent le lit : étiquette, titre, renvois, puis texte."""
    renvois = f" — renvois : {', '.join(r.codes_cites)}" if r.codes_cites else ""
    return f"[{etiquette}] {titre_resultat(r)}{renvois}\n{r.texte}"

## 4. Le prompt de l'agent

Un prompt **dédié à la recherche**, distinct de `systeme_vf.md` : l'agent ne répond pas au joueur. Il deviendra `agent_vf.md` dans le code propre ; modifie-le ici pour tes essais.

Points clés :

- **son rôle** : trouver tout ce qu'il faut pour répondre, et seulement ça ;
- **traduire** le vocabulaire du joueur ;
- **penser aux exceptions**, souvent rangées ailleurs que la règle générale ;
- **finir** par `retenir_passages`, dans un budget donné.

In [17]:
PROMPT_AGENT = f"""Tu es l'agent de recherche d'un assistant d'arbitrage pour le jeu Warhammer 40,000 (11e édition).
Tu ne réponds pas au joueur : tu assistes un autre modèle qui rédigera la réponse, uniquement à partir des passages que tu retiendras.
Ton travail est de trouver dans le livre de règles de base tous les passages nécessaires pour répondre complètement
et correctement à la question, et seulement ceux-là.

Méthode :
1. Identifie la ou les règles en jeu. Les joueurs n'emploient pas toujours le vocabulaire du livre : traduis leurs
   termes en termes probables du livre (noms de phases et d'actions, mots-clés en MAJUSCULES).
2. Cherche avec rechercher_regles, une requête par règle concernée. Si les résultats ne sont pas pertinents,
   reformule autrement. Effectue plusieurs recherches : lance une première recherche, analyse les résultats, puis effectue d'autres recherches.
3. Suis avec lire_regle les renvois utiles indiqués dans les passages.
4. Pense aux exceptions et aux cas particuliers : une règle générale a souvent des restrictions ailleurs dans le livre.
5. Quand tu as tout ce qu'il faut, appelle retenir_passages avec les étiquettes des passages utiles, du plus au moins
   important, au maximum {MAX_PASSAGES}. N'utilise que des étiquettes que tu as vues.

Les questions hors sujet ne doivent pas recevoir de réponse.Si aucun passage du livre ne concerne la question, il s'agit probablement d'une question hors sujet : appelle retenir_passages avec une liste vide.
Appuies-toi uniquement sur les informations contenues dans la base de données desrègles.
Tu disposes d'au plus {MAX_TOURS - 1} tours de recherche : sois efficace, regroupe les appels indépendants dans
un même tour.
"""
print(PROMPT_AGENT)

Tu es l'agent de recherche d'un assistant d'arbitrage pour le jeu Warhammer 40,000 (11e édition).
Tu ne réponds pas au joueur : tu assistes un autre modèle qui rédigera la réponse, uniquement à partir des passages que tu retiendras.
Ton travail est de trouver dans le livre de règles de base tous les passages nécessaires pour répondre complètement
et correctement à la question, et seulement ceux-là.

Méthode :
1. Identifie la ou les règles en jeu. Les joueurs n'emploient pas toujours le vocabulaire du livre : traduis leurs
   termes en termes probables du livre (noms de phases et d'actions, mots-clés en MAJUSCULES).
2. Cherche avec rechercher_regles, une requête par règle concernée. Si les résultats ne sont pas pertinents,
   reformule autrement. Effectue plusieurs recherches : lance une première recherche, analyse les résultats, puis effectue d'autres recherches.
3. Suis avec lire_regle les renvois utiles indiqués dans les passages.
4. Pense aux exceptions et aux cas particuliers : une

## 5. La boucle de recherche

À chaque tour, l'agent peut demander **plusieurs** outils à la fois. On les exécute et on renvoie leurs résultats.

La boucle s'arrête :

- **normalement**, quand l'agent appelle `retenir_passages` ;
- **par repli**, s'il répond en texte sans sélectionner, ou s'il épuise son budget de tours. Le code retient alors les passages vus, dans l'ordre, dans la limite de `MAX_PASSAGES`. Ça arrive rarement, mais le code ne doit jamais rester sans sélection.

Un tour avant la fin du budget, un message lui rappelle qu'il doit conclure. On ne force pas l'outil (`tool_choice`) : les modèles les plus récents refusent cette option.

Une étiquette inventée est ignorée avec un avertissement dans la trace.

In [11]:
@dataclass
class Selection:
    """Résultat de l'agent de recherche."""
    passages: list                      # Resultat retenus, dans l'ordre choisi
    justification: str = ""
    fin: str = ""                       # retenue | repli_texte | repli_budget
    appels: list = field(default_factory=list)   # (outil, argument, résultat résumé)
    vus: int = 0
    tours: int = 0
    tokens_entree: int = 0
    tokens_sortie: int = 0
    duree: float = 0.0
    modele: str = ""

    def cout(self):
        prix_entree, prix_sortie = TARIFS.get(self.modele, (0, 0))
        return (self.tokens_entree * prix_entree + self.tokens_sortie * prix_sortie) / 1e6


def chercher(question, modele=None, effort="defaut", max_tours=None, amorce=None):
    """Agent de recherche : renvoie une Selection."""
    modele = modele or MODELE_AGENT
    effort = EFFORT_AGENT if effort == "defaut" else effort
    max_tours = max_tours or MAX_TOURS
    amorce = AMORCE if amorce is None else amorce

    sel = Selection(passages=[], modele=modele)
    etiquettes, par_id = {}, {}          # étiquette -> Resultat ; id du chunk -> étiquette
    debut = time.perf_counter()

    def etiqueter(resultats):
        """Blocs texte pour l'agent : nouveaux passages en entier, déjà vus en rappel."""
        blocs, deja = [], []
        for r in resultats:
            if r.id in par_id:
                deja.append(par_id[r.id])
                continue
            e = f"P{len(etiquettes) + 1}"
            etiquettes[e], par_id[r.id] = r, e
            blocs.append({"type": "text", "text": texte_passage(e, r)})
        if deja:
            blocs.append({"type": "text", "text": f"Déjà fournis plus haut : {', '.join(deja)}."})
        return blocs or [{"type": "text", "text": "Aucun passage trouvé."}]

    contenu = []
    if amorce:
        initiaux = moteur.rechercher(question, K_RECHERCHE)
        contenu += [{"type": "text", "text": "Passages trouvés par une première recherche avec la question brute :"}]
        contenu += etiqueter(initiaux)
        sel.appels.append(("amorce", question, [r.code for r in initiaux]))
    contenu.append({"type": "text", "text": f"Question du joueur : {question}"})
    messages = [{"role": "user", "content": contenu}]

    for tour in range(1, max_tours + 1):
        options = {"output_config": {"effort": effort}} if effort else {}
        reponse = client.messages.create(model=modele, max_tokens=MAX_TOKENS_AGENT, system=PROMPT_AGENT,
                                         tools=OUTILS, messages=messages, **options)
        sel.tours = tour
        sel.tokens_entree += reponse.usage.input_tokens
        sel.tokens_sortie += reponse.usage.output_tokens
        messages.append({"role": "assistant", "content": reponse.content})

        appels_outils = [b for b in reponse.content if b.type == "tool_use"]
        fin = next((b for b in appels_outils if b.name == "retenir_passages"), None)
        if fin is not None:
            demandees = list(dict.fromkeys(fin.input.get("etiquettes", [])))   # sans doublons, ordre gardé
            inconnues = [e for e in demandees if e not in etiquettes]
            if inconnues:
                sel.appels.append(("retenir_passages", "étiquettes inconnues ignorées", inconnues))
            sel.passages = [etiquettes[e] for e in demandees if e in etiquettes][:MAX_PASSAGES]
            sel.justification = fin.input.get("justification", "")
            sel.fin = "retenue"
            sel.appels.append(("retenir_passages", ", ".join(demandees) or "(aucun)",
                               [r.code for r in sel.passages]))
            break
        if not appels_outils:   # l'agent a répondu en texte au lieu de sélectionner
            sel.fin = "repli_texte"
            break
        if tour == max_tours:   # budget épuisé : ses derniers appels ne seraient jamais lus
            sel.fin = "repli_budget"
            break

        resultats_outils = []
        for bloc in appels_outils:
            try:
                if bloc.name == "rechercher_regles":
                    trouves = moteur.rechercher(bloc.input["requete"], int(bloc.input.get("k", K_RECHERCHE)))
                    argument = bloc.input["requete"]
                elif bloc.name == "lire_regle":
                    trouves = lire_regle(bloc.input["code"])
                    argument = bloc.input["code"]
                else:
                    raise ValueError(f"outil inconnu : {bloc.name}")
                sel.appels.append((bloc.name, argument, [r.code for r in trouves]))
                resultats_outils.append({"type": "tool_result", "tool_use_id": bloc.id, "content": etiqueter(trouves)})
            except Exception as e:   # l'erreur est renvoyée à l'agent, qui peut corriger son appel
                sel.appels.append((bloc.name, str(bloc.input), [f"erreur : {e}"]))
                resultats_outils.append({"type": "tool_result", "tool_use_id": bloc.id,
                                         "content": f"Erreur : {e}", "is_error": True})
        if tour == max_tours - 1:
            resultats_outils.append({"type": "text", "text":
                "Budget de recherche épuisé : appelle maintenant retenir_passages avec les passages utiles."})
        messages.append({"role": "user", "content": resultats_outils})

    if sel.fin.startswith("repli"):
        sel.passages = list(etiquettes.values())[:MAX_PASSAGES]
    sel.vus = len(etiquettes)
    sel.duree = time.perf_counter() - debut
    return sel


def afficher_selection(sel):
    print(f"--- agent {sel.modele} : fin={sel.fin}, {sel.tours} tour(s), {sel.vus} passages vus, "
          f"{len(sel.passages)} retenus, {sel.duree:.1f} s, {sel.tokens_entree} + {sel.tokens_sortie} tokens, "
          f"{sel.cout() * 100:.2f} c$")
    for outil, argument, resultat in sel.appels:
        print(f"  {outil:>17}  « {argument} »  ->  {resultat}")
    if sel.justification:
        print(f"  justification : {sel.justification}")

## 6. La chaîne complète : recherche, puis génération

Le générateur reçoit la question d'origine et les passages retenus, **rien d'autre**. La justification de l'agent n'est pas transmise : c'est son raisonnement, pas un extrait du livre, et le générateur pourrait s'en servir comme d'une source.

**Sélection vide** : l'agent estime que rien dans le livre ne concerne la question. La phrase d'abstention est alors renvoyée directement, sans appeler le générateur. C'est un choix à observer : si l'agent se trompe, la question reste sans réponse alors que le livre la couvrait. L'alternative serait de transmettre quand même les passages de l'amorce au générateur, et de le laisser décider.

In [12]:
def repondre(question, **options_agent):
    """Recherche par l'agent puis génération ; renvoie (Reponse ou None, Selection)."""
    sel = chercher(question, **options_agent)
    if not sel.passages:
        return None, sel   # abstention sans appel au générateur
    return generateur.generer(question, sel.passages), sel


def afficher_tout(question, reponse, sel):
    print("=" * 100, "\nQUESTION :", question, "\n" + "=" * 100)
    afficher_selection(sel)
    print()
    print(formater_reponse(reponse) if reponse else G.phrase_abstention + "  (sélection vide, générateur non appelé)")

## 7. Une question, en détail

Choisis une question **en langage de joueur**. Le plus instructif est la trace : les **requêtes écrites par l'agent** montrent s'il fait le lien avec le vocabulaire du livre, et sa **sélection** montre s'il a pensé aux exceptions.

In [14]:
question = "Je dois faire quoi avec mes gars qui fuient?"   # à adapter avec un vocabulaire loin de celui du livre mais un sens proche

question = "Mon perso peut-il rejoindre une escouade pour la protéger ?"
reponse, sel = repondre(question)
afficher_tout(question, reponse, sel)

QUESTION : Mon perso peut-il rejoindre une escouade pour la protéger ? 
--- agent claude-sonnet-5 : fin=retenue, 3 tour(s), 17 passages vus, 6 retenus, 14.5 s, 20611 + 972 tokens, 5.09 c$
             amorce  « Mon perso peut-il rejoindre une escouade pour la protéger ? »  ->  ['19.01', None, '03.01', None, '19.02']
  rechercher_regles  « cibler un PERSONNAGE dans une unité attachée protection attribution des blessures »  ->  ['24.28', '19.02', '13.08', '22.05', '17.03']
  rechercher_regles  « unité de gardes du corps unité de meneur restrictions PERSONNAGE seul »  ->  ['19.01', '19.04', '19.04', '19.04', '02.06']
  rechercher_regles  « ordre d'allocation groupe d'allocation blessures figurine la plus proche »  ->  ['05.04', '24.28', '05.03', '05.04', '05.04']
   retenir_passages  « P1, P2, P15, P5, P6, P11 »  ->  ['19.01', None, '05.03', '19.02', '24.28', '19.04']
  justification : P1/P2 expliquent la formation des unités attachées (un personnage meneur/appui peut rejoindre une escoua

## 8. Baseline contre deux étapes (même générateur)

Pour chaque question :

- **baseline** : top-k de la question brute, puis générateur ;
- **deux étapes** : sélection de l'agent, puis le **même** générateur.

Avec les **codes attendus**, on vérifie s'ils sont dans le contexte **fourni** au générateur (le retrieval les a-t-il trouvés ?) et dans les passages **cités** (la réponse s'appuie-t-elle dessus ?).

In [15]:
QUESTIONS = {
    # "question en langage de joueur": ["codes", "attendus"],   (liste vide si inconnue)
    "Je dois faire quoi avec mes gars qui fuient?" : ["09.07", "06.03","01.07","09.07"],
    "Mes gars ont couru ce tour, est-ce qu'ils peuvent encore tirer ?": ["10.04","10.05","09.06"],
    "Je peux sortir du corps à corps et tirer dans la foulée ?": ["09.07"],
    "Mon perso peut-il rejoindre une escouade pour la protéger ?": ["19.01"],
    "Quelle est la recette de la pâte à crêpes ?": [],   # hors corpus : l'abstention doit tenir
}


def verdict(attendus, codes):
    return "-" if not attendus else ("oui" if set(attendus) & set(codes) else "NON")


resultats_comparaison = {}
print(f"{'méthode':>11} | {'fournis':>7} | {'trouvé':>6} | {'cité':>5} | {'durée':>6} | {'coût':>8} | codes fournis")
for question, attendus in QUESTIONS.items():
    print(f"\n{question}")
    passages = moteur.rechercher(question, config.recherche.k)
    rep_b = generateur.generer(question, passages)
    cout_b = (rep_b.tokens_entree * TARIFS[G.modele][0] + rep_b.tokens_sortie * TARIFS[G.modele][1]) / 1e6
    fournis_b = [r.code for r in passages]
    print(f"{'baseline':>11} | {len(fournis_b):>7} | {verdict(attendus, fournis_b):>6} | "
          f"{verdict(attendus, [c.code for c in rep_b.citations]):>5} | {rep_b.duree:5.1f}s | "
          f"{cout_b * 100:6.2f}c$ | {fournis_b}")

    rep_a, sel = repondre(question)
    fournis_a = [r.code for r in sel.passages]
    cites_a = [c.code for c in rep_a.citations] if rep_a else []
    duree_a = sel.duree + (rep_a.duree if rep_a else 0)
    cout_a = sel.cout() + ((rep_a.tokens_entree * TARIFS[G.modele][0] + rep_a.tokens_sortie * TARIFS[G.modele][1]) / 1e6
                           if rep_a else 0)
    print(f"{'2 étapes':>11} | {len(fournis_a):>7} | {verdict(attendus, fournis_a):>6} | "
          f"{verdict(attendus, cites_a):>5} | {duree_a:5.1f}s | {cout_a * 100:6.2f}c$ | {fournis_a}")
    resultats_comparaison[question] = (rep_b, rep_a, sel)

    méthode | fournis | trouvé |  cité |  durée |     coût | codes fournis

Je dois faire quoi avec mes gars qui fuient?
   baseline |       5 |    NON |   NON |   7.2s |   1.61c$ | ['05.04', '05.03', '05.04', '05.04', '04.03']
   2 étapes |       3 |    oui |   oui |  23.0s |   7.85c$ | ['09.07', '01.07', '06.03']

Mes gars ont couru ce tour, est-ce qu'ils peuvent encore tirer ?
   baseline |       5 |    oui |   oui |   7.3s |   1.60c$ | ['15.09', '10.07', '10.04', '10.05', None]
   2 étapes |       3 |    oui |   oui |  14.1s |   3.71c$ | ['10.05', '10.04', '09.06']

Je peux sortir du corps à corps et tirer dans la foulée ?
   baseline |       5 |    NON |   NON |  15.1s |   2.27c$ | [None, '17.03', '24.27', '04.02', '10.02']
   2 étapes |       4 |    oui |   oui |  17.6s |   6.62c$ | ['09.07', '10.04', '04.02', '10.05']

Mon perso peut-il rejoindre une escouade pour la protéger ?
   baseline |       5 |    oui |   oui |  13.8s |   2.10c$ | ['19.01', None, '03.01', None, '19.02']
 

In [16]:
# Détail d'une question : les deux réponses et la trace de l'agent
question = list(QUESTIONS)[0]   # change l'indice
rep_b, rep_a, sel = resultats_comparaison[question]

print("########## BASELINE ##########")
print(formater_reponse(rep_b))
print("\n########## DEUX ÉTAPES ##########")
afficher_tout(question, rep_a, sel)

########## BASELINE ##########
Je ne trouve pas la réponse dans les extraits du livre de règles fournis.

Les extraits dont je dispose concernent uniquement la séquence d'attaque (04.03 Résoudre les attaques, 05.03 Jets de sauvegarde et 05.04 Infliger des dégâts). Ils ne traitent pas de la notion de figurines qui « fuient » (ce qui pourrait renvoyer à des règles de Battle-shock, de retraite ou de Desperate Escape selon les éditions).

Pourrais-tu préciser ta question ? Par exemple :
- s'agit-il d'une unité qui a subi un test de Battle-shock et dont tu veux connaître les conséquences,
- ou d'une unité qui doit quitter le combat (retraite) suite à une charge,
- ou encore d'une situation liée à un Transport détruit (Desperate Escape) ?

Avec ces précisions, je pourrai rechercher les sections pertinentes du livre de règles pour te répondre correctement.

(claude-sonnet-5, prompt systeme_vf.md [e270d337], 7.2 s, 5286 tokens en entrée, 549 en sortie, 0% du texte cité)

########## DEUX ÉTAPES

## 9. Quel modèle pour l'agent ?

On compare **uniquement la sélection** (sans génération) : c'est ce qui change d'un modèle à l'autre, et ça coûte moins cher. Pour chaque modèle : codes attendus trouvés, nombre de tours, passages vus et retenus, durée, coût.

Ce qui compte pour choisir :

- la **qualité de la sélection** : les bons codes, les exceptions pensées. Lis les sélections, pas seulement les chiffres ;
- la **latence** : le joueur attend toute la recherche avant que la réponse commence à s'afficher ;
- le **coût** par question.

Si un modèle refuse le paramètre d'effort (erreur 400), remplace son effort par `None`.

In [ ]:
MODELES_AGENT = [
    ("claude-haiku-4-5-20251001", None),   # effort non envoyé : à vérifier pour ce modèle
    ("claude-sonnet-5", "high"),
    ("claude-opus-5-5", "high"),
]

bilan = {m: [] for m, _ in MODELES_AGENT}
for question, attendus in QUESTIONS.items():
    print(f"\n{question}")
    for modele, effort in MODELES_AGENT:
        sel = chercher(question, modele=modele, effort=effort)
        codes = [r.code for r in sel.passages]
        bilan[modele].append(sel)
        print(f"  {modele:>26} | {verdict(attendus, codes):>6} | {sel.tours} tours | {sel.vus:>2} vus | "
              f"{len(codes)} retenus | {sel.duree:5.1f}s | {sel.cout() * 100:5.2f}c$ | {sel.fin:>12} | {codes}")

print(f"\n{'modèle':>28} | {'durée moy.':>10} | {'coût moy.':>9} | replis")
for modele, selections in bilan.items():
    n = len(selections)
    replis = sum(s.fin != "retenue" for s in selections)
    print(f"{modele:>28} | {sum(s.duree for s in selections) / n:9.1f}s | "
          f"{sum(s.cout() for s in selections) / n * 100:7.2f}c$ | {replis}/{n}")

In [ ]:
# Traces complètes d'une question pour chaque modèle : comparer les requêtes et les sélections
question = list(QUESTIONS)[0]   # change l'indice
i = list(QUESTIONS).index(question)
for modele, _ in MODELES_AGENT:
    print()
    afficher_selection(bilan[modele][i])

## Bilan : décisions pour le code propre

- **modèle et effort de l'agent** (section 9) : le compromis entre qualité de sélection, latence et coût ;
- **amorce** : utile, ou l'agent trouve-t-il seul ?
- **budget** : `MAX_TOURS` et `MAX_PASSAGES` suffisants ? Des replis (fin ≠ « retenue ») ?
- **sélection vide** : abstention directe, ou transmettre l'amorce au générateur ?
- **les requêtes de l'agent** : traduisent-elles le vocabulaire ? Sinon, retravailler le prompt ou la description des outils ;
- **les questions** de la section 8, avec leurs codes attendus, pour le gold set (catégorie « vocabulaire joueur »).

In [ ]:
conn.close()